# *Embeddings Contextuales*

### **BERT**

BERT utiliza una arquitectura Transformer bidireccional, no LSTM, lo que permite una comprensión profunda de las relaciones semánticas y sintácticas. Usaremos el modelo bert-base-multilingual-cased para obtener representaciones de 768 dimensiones para cada noticia como un caso basico y generico, el cual posria ser de utilidad para nuestra tarea de generacion de una frase que resuma el texto.

In [2]:
from transformers import AutoModel, AutoTokenizer
import torch
import numpy as np

MODEL_NAME = "bert-base-multilingual-cased" # Mismo modelo que el tokenizador

model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)

c:\Users\Iñigo Peña\Desktop\FinTracker\ftenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tenemos que cargar los datos preprocesados. Estos datos deben estar en el formato que el modelo espera: IDs de tokens (input_ids) y una *attention mask* que indica qué tokens son reales y cuáles son padding.

In [2]:
import pandas as pd

textos_tokenizados_bert = pd.read_parquet("datasPost_prepro/bertTokenized.parquet", engine='fastparquet')
print(textos_tokenizados_bert.head())

                                        article_text  \
0  The UK jobs market continues to show signs of ...   
1  New data from the Department from Work and Pen...   
2  Asking for workplace accommodations is often e...   
3  Apple has announced a major expansion of its r...   
4  The advent of artificial intelligence (AI) les...   

                                           input_ids  \
0  [101, 10117, 10523, 45083, 17313, 25266, 10114...   
1  [101, 10287, 11165, 10188, 10105, 12933, 10188...   
2  [101, 93919, 10230, 10142, 11424, 30236, 10789...   
3  [101, 17216, 10393, 13854, 169, 11922, 24837, ...   
4  [101, 10117, 10840, 22657, 10108, 36866, 30151...   

                                      attention_mask  
0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
1  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
2  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
3  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
4  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..

Para que el modelo BERT pueda procesar los datos, necesitamos convertir las listas de IDs a tensores, ya que son la estrucutra de datos nativa de PyTorch.

In [8]:
try:
    textos_tokenizados_bert = pd.read_parquet("datasPost_prepro/bertTokenized.parquet", engine='fastparquet')

    # Convertir las columnas a listas
    input_ids_list = textos_tokenizados_bert['input_ids'].tolist()
    attention_mask_list = textos_tokenizados_bert['attention_mask'].tolist()

    # Convertir las listas a tensores de PyTorch
    input_ids = torch.tensor(input_ids_list)
    attention_mask = torch.tensor(attention_mask_list)

    print("Datos cargados con fastparquet.")
    print("input_ids:\n", input_ids)
    print("attention_mask:\n", attention_mask)
except Exception as e:
    print(f"Error al usar fastparquet: {e}")

Datos cargados con fastparquet.
input_ids:
 tensor([[  101, 10117, 10523,  ...,     0,     0,     0],
        [  101, 10287, 11165,  ..., 10747, 10944,   102],
        [  101, 93919, 10230,  ..., 12166, 19573,   102],
        ...,
        [  101, 46291, 11281,  ...,     0,     0,     0],
        [  101, 15940,   100,  ..., 61093, 13028,   102],
        [  101, 12489, 10944,  ...,   112,   187,   102]])
attention_mask:
 tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])


El modelo nos daba problemas de capacidad en la memoria. Si intentáramos procesar todas las noticias a la vez, el sistema quedaba sin RAM (CPU) e incluso sin VRAM (GPU).

Para evitar fallos, implementamos un procesamiento por lotes (BATCH_SIZE = 32) y nos aseguramos de que solo se mueva a la memoria de la GPU o CPU el pequeño subconjunto de datos que se está utilizando en ese instante.

In [ ]:
# GPU-CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BATCH_SIZE = 32

all_embeddings = []
num_batches = (len(input_ids) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(num_batches):
    start_idx = i * BATCH_SIZE
    end_idx = min((i + 1) * BATCH_SIZE, len(input_ids))

    batch_input = {
        'input_ids': input_ids[start_idx:end_idx],
        'attention_mask': attention_mask[start_idx:end_idx]
    }
    
    # Mover batch a GPU-CPU
    batch_input['input_ids'] = batch_input['input_ids'].to(device)
    batch_input['attention_mask'] = batch_input['attention_mask'].to(device)

    # Embedding para batch (desactiva el cálculo de gradientes)
    with torch.no_grad():
        model_output = model(**batch_input) # Lote pasa por BERT, generando los embeddings

    batch_cls_embeddings = model_output.last_hidden_state[:, 0, :].cpu().numpy()
    all_embeddings.append(batch_cls_embeddings)

# Concatenar todos los embeddings
final_embeddings = np.concatenate(all_embeddings, axis=0)
print(f"Embeddings finales generados: {final_embeddings.shape}")

Embeddings finales generados: (5160, 768)


El código itera sobre todos los textos, dividiendo los tensores grandes (input_ids, attention_mask) en trozos de tamaño BATCH_SIZE; es decir, procesa los textos de 32 en 32. Dentro del bucle, en las líneas *batch_input['input_ids'] = ... .to(device)* y *batch_input['attention_mask'] = ... .to(device)* solo movemos el lote actual (32 noticias) a la memoria de la GPU justo antes de la inferencia, evitando saturarla con todo el dataset.

In [ ]:
import pandas as pd
import numpy as np

# Creamos una lista de nombres de columna para las 768 dimensiones (opcional, pero ayuda)
num_dimensions = final_embeddings.shape[1]
column_names = [f'dim_{i}' for i in range(num_dimensions)]

# Convertir el array de NumPy a un DataFrame
df_embeddings = pd.DataFrame(final_embeddings, columns=column_names)


In [13]:
print("Estructura del DataFrame de Embeddings:")
print(df_embeddings.head(2))
print("\nInformación del DataFrame:")
df_embeddings.info()

Estructura del DataFrame de Embeddings:
      dim_0     dim_1     dim_2     dim_3     dim_4     dim_5     dim_6  \
0 -0.248673  0.100390 -0.130808 -0.133665  0.263254 -0.177622 -0.247014   
1  0.099093 -0.072926 -0.091603 -0.036458 -0.153845 -0.079636 -0.143765   

      dim_7     dim_8     dim_9  ...   dim_758   dim_759   dim_760   dim_761  \
0  0.148019 -0.171507  0.234010  ...  0.287633 -0.078372 -0.151943 -0.237940   
1  0.026207  0.136078  0.081696  ...  0.312628 -0.196120  0.245679 -0.038302   

    dim_762   dim_763   dim_764   dim_765   dim_766   dim_767  
0 -0.054363  0.185864 -0.055884  0.114988  0.140016 -0.051910  
1 -0.236432  0.100330  0.135540  0.422602  0.030791  0.117918  

[2 rows x 768 columns]

Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5160 entries, 0 to 5159
Columns: 768 entries, dim_0 to dim_767
dtypes: float32(768)
memory usage: 15.1 MB


In [11]:
# Definir la ruta y el nombre del archivo
PARQUET_FILE_PATH = 'embeddings/bert_embeddings.parquet'

# Guardar en formato Parquet
df_embeddings.to_parquet(PARQUET_FILE_PATH, engine='fastparquet', index=False)

print(f"Embeddings guardados exitosamente en: {PARQUET_FILE_PATH}")

Embeddings guardados exitosamente en: embeddings/bert_embeddings.parquet


### Huggin Face **Fin-BERT**

Ahora vamos a crear la representacion en embeddings con el modelo que utilizaremos en la tarea de reconocimiento de entidades (NER). Para ello vamos a utilizar los input IDs y attention masks extraidos en el preprocesamiento.

In [5]:
tokens_df = pd.read_parquet("datasPost_prepro/fin-bertTokenized.parquet", engine='fastparquet')
print(tokens_df.head(2))

                                        article_text  \
0  The UK jobs market continues to show signs of ...   
1  New data from the Department from Work and Pen...   

                                           input_ids  \
0  [101, 1996, 2866, 5841, 3006, 4247, 2000, 2265...   
1  [101, 2047, 2951, 2013, 1996, 2533, 2013, 2147...   

                                      attention_mask  
0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
1  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  


Aqui ya podemos apreciar como el vocabulario de este modelo es diferente al mas general que hemos utilizado en el ejemplo anterior, ya que los IDs de las mismas palabras no coinciden.

In [6]:
try:
    # Convertir las columnas a listas
    input_ids_list = tokens_df['input_ids'].tolist()
    attention_mask_list = tokens_df['attention_mask'].tolist()

    # Convertir las listas a tensores de PyTorch
    input_ids = torch.tensor(input_ids_list)
    attention_mask = torch.tensor(attention_mask_list)

    print("Datos cargados con fastparquet.")
    print("input_ids:\n", input_ids)
    print("attention_mask:\n", attention_mask)
except Exception as e:
    print(f"Error al usar fastparquet: {e}")

Datos cargados con fastparquet.
input_ids:
 tensor([[  101,  1996,  2866,  ...,     0,     0,     0],
        [  101,  2047,  2951,  ...,  2077,  2037,   102],
        [  101,  4851,  2005,  ...,  5005,  1011,   102],
        ...,
        [  101,  7211,  1006,  ...,     0,     0,     0],
        [  101,  5148,  1521,  ..., 29336,  5369,   102],
        [  101,  2054,  2064,  ...,  2028,  1011,   102]])
attention_mask:
 tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

Para esta tarea vamos a aplicar layer pooling para el output, usando la estrategia 'last4_mean', en la que se calcula una medi de las ultimas 4 capas del modelo Transformer.

In [8]:
import torch

def last4_layer_pool(outputs):
    last4 = torch.stack(outputs.hidden_states[-4:], dim=0)  # (4, B, T, H)
    return last4.mean(dim=0)  # (B, T, H) 


Una vez más, realizaremos un bucle organizado en lotes de 32 textos (BATCH_SIZE=32) y moveremos cada lote a la memoria de la GPU justo antes de la inferencia, evitando saturarla con todo el dataset.

In [ ]:
import math
import numpy as np

# input_ids y attention_mask
assert input_ids.shape == attention_mask.shape
N, T = input_ids.shape

BATCH_SIZE = 32
num_batches = math.ceil(N / BATCH_SIZE)

all_embeds = []  # lista de arrays (len = tokens reales de cada doc, dim = H)

with torch.no_grad():
    for i in range(num_batches):
        start = i * BATCH_SIZE
        end = min((i + 1) * BATCH_SIZE, N)

        ids = input_ids[start:end].to(device)
        mask = attention_mask[start:end].to(device)

        outputs = model(input_ids=ids, attention_mask=mask)
        tok_emb = last4_layer_pool(outputs)  # (B, T, H)

        # 
        tok_emb = tok_emb.cpu()
        mask_cpu = mask.cpu()

        # separar por documento, y recortar padding con la attention_mask
        for b in range(tok_emb.size(0)):
            valid_len = int(mask_cpu[b].sum().item())  # nº de tokens reales (sin PAD)
            emb_np = tok_emb[b, :valid_len, :].numpy() # (valid_len, H)
            all_embeds.append(emb_np)


In [53]:
print("Embeddings de cada token de las primeras 2 noticias:\n")
print("Texto 1:\n", all_embeds[0])
print("Texto 2:\n", all_embeds[1])

Embeddings de cada token de las primeras 2 noticias:

Texto 1:
 [[ 0.07919598  0.42304555  0.49296823 ... -0.19378875  0.24579032
  -1.9666685 ]
 [-0.52735704  0.5634529   0.6335957  ...  1.0225722   0.68728304
   0.6458372 ]
 [ 0.52096206 -0.12409208  0.34956995 ... -0.21167994  1.3996931
   0.1240859 ]
 ...
 [ 0.01897943  0.17833304  0.6456697  ... -0.38537624  0.3641638
   0.42148378]
 [ 0.08578334  1.2148362   0.72271925 ... -0.8737022   0.32135978
   0.3613027 ]
 [-0.8424122   0.07099887  0.8274948  ...  0.01402965  0.06313592
  -0.0844599 ]]
Texto 2:
 [[ 0.03193333  0.5028747   0.5432351  ... -0.02114106  0.23125128
  -1.9135686 ]
 [ 0.668228    1.0169243   0.82296723 ... -0.11314092  0.53316087
   0.22302602]
 [-1.2144669   2.4859974   0.33820406 ... -0.48560676  1.1921617
   1.4286529 ]
 ...
 [ 0.7785916  -0.6181889  -0.2850606  ... -0.9233468   0.70517325
   0.1907533 ]
 [ 0.39720193  0.5653375  -0.11222406 ... -0.49287596  0.3944539
   1.170919  ]
 [ 0.6684627   0.17865683  0

Ahora all_embeds es una lista con un array por documento, y cada fila de estos arrays representa el embedding de un subtoken real, siendo la dimension de cada embedding de 768 valores.

In [ ]:
import pandas as pd

rows = []
for doc_id, arr in enumerate(all_embeds):
    rows.append({
        "doc_id": doc_id,
        "num_tokens": int(arr.shape[0]),
        "embedding_dim": int(arr.shape[1]) if arr.size else 0,
        "subtoken_embeddings": [v.tolist() for v in arr]   # lista de listas (tokens × H)
    })

df_emb = pd.DataFrame(rows)
df_emb.to_parquet("embeddings/fin-bert_embeddings.parquet", index=False)
